In [1]:
!pip install openai gradio

In [ ]:
import os
import csv
from datetime import datetime
import gradio as gr
from openai import OpenAI

# Set your OpenAI API key here. Replace 'YOUR_OPENAI_API_KEY' with your actual key.
# You can also set it as an environment variable in Colab secrets or your system.
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 🧠 Categorize user input
def categorize_input(user_input: str) -> str:
    user_input = user_input.lower()
    if "anxious" in user_input:
        return "anxious"
    elif "focus" in user_input:
        return "focus"
    elif "overwhelmed" in user_input:
        return "overwhelmed"
    else:
        return "default"

# 📝 Log interactions
def log_interaction(user_input: str, response: str, log_file="logs.csv"):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(log_file, mode="a", newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow([timestamp, user_input, response])

# 🤖 Generate motivation using LLM
def generate_motivation(user_input: str) -> str:
    category = categorize_input(user_input)

    prompt = f"""
    You are a kind and supportive motivational coach.

    User mood: {category}
    User says: "{user_input}"

    Instructions:
    - Be empathetic and encouraging
    - Keep response short (2-3 lines)
    - Make it feel personal
    - Avoid generic clichés

    Give a motivational message:
    """

    try:
        # The `responses.create` method is not standard for OpenAI Python client.
        # It should be `client.chat.completions.create` for chat models
        # or `client.completions.create` for older completion models.
        # Assuming `gpt-5-mini` is a chat-based model, the correct call should be:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",  # Using a common chat model as an example
            messages=[
                {"role": "system", "content": "You are a kind and supportive motivational coach."},
                {"role": "user", "content": prompt}
            ]
        )

        text = response.choices[0].message.content

    except Exception as e:
        # More specific error handling could be added here
        text = f"⚠️ Error generating response: {e}. Check API key, model name, or internet connection."

    log_interaction(user_input, text)
    return text

# 🎨 Gradio UI
with gr.Blocks(theme=gr.themes.Base(primary_hue="cyan", secondary_hue="violet")) as demo:

    gr.Markdown("""
    # 💬 AI Daily Motivation Generator
    Feeling low or distracted? Let AI lift your spirits ✨
    """)

    with gr.Row():
        with gr.Column(scale=1):
            user_input = gr.Textbox(
                label="🌈 What's your mood today?",
                placeholder="e.g., feeling anxious, need focus...",
                lines=2
            )
            submit_btn = gr.Button("✨ Generate Motivation")

        with gr.Column(scale=1):
            output = gr.Textbox(
                label="🧘 Your Personalized Motivation",
                lines=4,
                interactive=False,
                show_copy_button=True
            )

    submit_btn.click(fn=generate_motivation, inputs=user_input, outputs=output)

    gr.Examples(
        examples=[
            ["feeling anxious"],
            ["need focus for work"],
            ["feeling overwhelmed"],
            ["preparing for an interview"]
        ],
        inputs=user_input
    )

# 📁 Create log file if not exists
if __name__ == "__main__":
    if not os.path.exists("logs.csv"):
        with open("logs.csv", mode="w", newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Timestamp", "User Input", "Response"])

    demo.launch(share=True, debug=True)
